# Feature engineering, folds, and peak-risk threshold

Validation tests for three utilities added to `src/common/`: expanding-window folds
(`folds.py`), the per-fold peak-risk threshold (`peak_risk.py`), and lag/rolling/multi-output
feature engineering (`feature_engineering.py`). Loads `fsa_hourly_master_clean.parquet` and
checks each piece before training the LightGBM regressor.

In [19]:
# Jupyter's cwd is the notebook's folder, not the repo root, so "from src.common..." below
# needs the repo root on sys.path. This walks up parent folders until it finds requirements.txt
# (which only exists at the repo root) and adds that folder to sys.path.
import sys
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "requirements.txt").exists():
            return parent
    raise FileNotFoundError("Could not locate repo root (requirements.txt not found)")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT

WindowsPath('c:/workspace/CAPSTONE/ontario-electricity-peak-risk')

In [20]:
import pandas as pd

from src.common.data_loading import INTERIM_DIR, save_interim_dataset
from src.common.feature_engineering import (
    add_lag_features,
    add_multi_output_targets,
    add_rolling_features,
)
from src.common.folds import iter_folds
from src.common.peak_risk import compute_peak_thresholds, label_peaks

pd.set_option("display.max_columns", 50)

## 1. Load the cleaned dataset

Loads `fsa_hourly_master_clean.parquet`, the interpolated output of the previous notebook.

In [21]:
dataset = pd.read_parquet(INTERIM_DIR / "fsa_hourly_master_clean.parquet")
dataset.shape

(262944, 54)

## 2. Expanding-window folds

3 folds (`src/common/folds.py`), each training on a growing set of full years and testing on
the year after. Confirms below that train and test never share a timestamp.

In [22]:
fold_summary = []
for fold, train, test in iter_folds(dataset):
    overlap = set(train["timestamp_local"]) & set(test["timestamp_local"])
    fold_summary.append(
        {
            "fold": fold.number,
            "train_years": fold.train_years,
            "test_year": fold.test_year,
            "train_rows": len(train),
            "test_rows": len(test),
            "overlap": len(overlap),
        }
    )
pd.DataFrame(fold_summary)

,fold,train_years,test_year,train_rows,test_rows,overlap
0,1,"(2021, 2022)",2023,105120,52560,0
1,2,"(2021, 2022, 2023)",2024,157680,52704,0
2,3,"(2021, 2022, 2023, 2024)",2025,210384,52560,0


## 3. Peak-risk threshold per fold

`compute_peak_thresholds` takes the 97.5th percentile of consumption per (`fsa`, `season`) from
the fold's training rows only, confirmed below by asserting they fall inside the fold's training
years. `label_peaks` applies that threshold to both splits: train lands near 2.5% by
construction, test is free to differ since it depends on the actual future consumption.

In [23]:
threshold_summary = []
for fold, train, test in iter_folds(dataset):
    assert train["timestamp_local"].dt.year.isin(fold.train_years).all()

    thresholds = compute_peak_thresholds(train)
    train_labeled = label_peaks(train, thresholds)
    test_labeled = label_peaks(test, thresholds)

    threshold_summary.append(
        {
            "fold": fold.number,
            "test_year": fold.test_year,
            "n_groups": len(thresholds),
            "train_peak_rate_pct": round(train_labeled["actual_peak"].mean() * 100, 2),
            "test_peak_rate_pct": round(test_labeled["actual_peak"].mean() * 100, 2),
        }
    )
pd.DataFrame(threshold_summary)

,fold,test_year,n_groups,train_peak_rate_pct,test_peak_rate_pct
0,1,2023,24,2.51,10.95
1,2,2024,24,2.50,4.00
2,3,2025,24,2.50,6.93


In [24]:
# thresholds from the last fold in the loop above, as an example of the (fsa, season) table
thresholds.sort_values(["fsa", "season"])

,fsa,season,peak_threshold
0,L4T,Fall,16401.8625
1,L4T,Spring,14691.6675
2,L4T,Summer,23057.0075
3,L4T,Winter,15430.0525
4,M5R,Fall,9817.3625
5,M5R,Spring,9986.9900
6,M5R,Summer,12535.8000
7,M5R,Winter,12079.3150
8,M5S,Fall,5310.4000
9,M5S,Spring,5383.5600


## 4. Lag and rolling consumption features

`add_lag_features` adds trailing lags (1h, 2h, 3h, 24h, 48h, 168h); `add_rolling_features` adds
trailing mean/std over 24h and 168h windows. Both grouped by `fsa`, windows ending at each row's
own hour since that value is already known at forecast time.

In [25]:
engineered = add_lag_features(dataset)
engineered = add_rolling_features(engineered)

new_columns = [
    c for c in engineered.columns
    if c.startswith("consumption_lag_") or c.startswith("consumption_roll_")
]
engineered[new_columns].isna().sum()

consumption_lag_1h               6
consumption_lag_2h              12
consumption_lag_3h              18
consumption_lag_24h            144
consumption_lag_48h            288
consumption_lag_168h          1008
consumption_roll_mean_24h      138
consumption_roll_std_24h       138
consumption_roll_mean_168h    1002
consumption_roll_std_168h     1002
dtype: int64

## 5. Multi-output targets

`add_multi_output_targets` adds `target_h1` through `target_h24`, the actual consumption n hours
after each row's timestamp, grouped by `fsa`. This is the wide `Y` matrix `MultiOutputRegressor`
needs, no separate pivot required. Confirmed below on M5S.

In [26]:
engineered = add_multi_output_targets(engineered)

target_columns = [c for c in engineered.columns if c.startswith("target_h")]
max_nan = engineered[target_columns].isna().sum().max()
print(len(target_columns), "target columns,", max_nan, "max NaN in a single column")

m5s = engineered.loc[engineered["fsa"] == "M5S"].sort_values("timestamp_local").reset_index(drop=True)
target_h1_matches_next_row = (
    m5s["target_h1"].iloc[:-1].to_numpy() == m5s["electricity_consumption"].iloc[1:].to_numpy()
).all()
print("target_h1 matches next row's consumption on M5S:", target_h1_matches_next_row)

24 target columns, 144 max NaN in a single column
target_h1 matches next row's consumption on M5S: True


## 6. Save the engineered dataset

Saves the full feature set to `data/interim/fsa_hourly_features.parquet`. Splitting by fold and
dropping rows with missing lags/targets is left for the training notebook, since that depends on
the fold.

In [27]:
output_path = save_interim_dataset(engineered, filename="fsa_hourly_features.parquet")
output_path

WindowsPath('C:/workspace/CAPSTONE/ontario-electricity-peak-risk/data/interim/fsa_hourly_features.parquet')